## Input · The message to analyse

In [ ]:
SENTENCE = "How are you with moneY...as in to you...money aint a thing....how are you sha!"
SENTENCE_INDEX = 0

## Setup · Imports and NLTK resources

In [ ]:
# ruff: noqa: E402
import math
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import nltk

from course_assessment.examples.selected_sentences import load_sms_split
from course_assessment.language_model.bigram_model import START_TOKEN, BigramLanguageModel
from course_assessment.preprocessing.text_preprocessor import (
    ensure_nltk_resources,
    lowercase_text,
    normalize_message,
    remove_numbers,
    remove_punctuation,
    remove_stop_words,
    segment_sentences,
    stem_tokens,
    tokenize_words,
    trace_message,
)
from course_assessment.run_assessment import build_training_sentences

ensure_nltk_resources()
print("Project root :", ROOT)
print("NLTK resources: OK")

## Dataset lookup · Is the message in the corpus?

In [ ]:
train_records, heldout_records, info = load_sms_split()
message = SENTENCE.strip()
match = None
for split_name, records in (("TRAINING", train_records), ("HELD-OUT", heldout_records)):
    for record in records:
        if record["text"] == message:
            match = (split_name, record)
            break
    if match:
        break

if match:
    split_name, record = match
    print(f"Found in dataset -> label: {record['label']} | row_id: {record['row_id']} | split: {split_name}")
else:
    print("Not found in the dataset (exact text match after strip). Continuing with the text as typed.")

## Part A · Stage 1 · Lowercasing

In [ ]:
lowered = lowercase_text(SENTENCE)
print("Original :", repr(SENTENCE))
print("Lowercase:", repr(lowered))

## Part A · Stage 2 · Sentence Segmentation

In [ ]:
sentences = segment_sentences(lowered)
print("Number of sentences:", len(sentences))
for i, sentence in enumerate(sentences):
    print(f"  [{i}] {sentence!r}")
if not sentences:
    print("  (the message is empty: nothing to process)")

## Part A · Stage 3 · Word Tokenization

In [ ]:
tokens_per_sentence = [tokenize_words(sentence) for sentence in sentences]
for i, tokens in enumerate(tokens_per_sentence):
    print(f"Sentence [{i}] ({len(tokens)} tokens): {tokens}")

## Part A · Stage 4 · Punctuation Removal

In [ ]:
trace = trace_message(SENTENCE)
no_punctuation = [remove_punctuation(tokens) for tokens in tokens_per_sentence]
for i, tokens in enumerate(no_punctuation):
    print(f"Sentence [{i}] result : {tokens}")
    print(f"             removed: {trace['sentence_traces'][i]['removed_as_punctuation']}")

## Part A · Stage 5 · Number Removal

In [ ]:
no_numbers = [remove_numbers(tokens) for tokens in no_punctuation]
for i, tokens in enumerate(no_numbers):
    print(f"Sentence [{i}] result : {tokens}")
    print(f"             removed: {trace['sentence_traces'][i]['removed_as_numbers']}")

## Part A · Stage 6 · Stop-Word Removal

In [ ]:
no_stop_words = [remove_stop_words(tokens) for tokens in no_numbers]
for i, tokens in enumerate(no_stop_words):
    print(f"Sentence [{i}] result : {tokens}")
    print(f"             removed: {trace['sentence_traces'][i]['removed_as_stop_words']}")

## Part A · Stage 7 · Porter Stemming

In [ ]:
stemmed = [stem_tokens(tokens) for tokens in no_stop_words]
for i, tokens in enumerate(stemmed):
    changes = trace["sentence_traces"][i]["stem_changes"]
    print(f"Sentence [{i}] result : {tokens}")
    if changes:
        print("             changed :", ", ".join(f"{word} -> {stem}" for word, stem in changes.items()))
    else:
        print("             changed : none")

## Part A · Final normalized tokens

In [ ]:
final_tokens = normalize_message(SENTENCE)
for i, tokens in enumerate(final_tokens):
    print(f"Sentence [{i}]: {tokens}" + ("   (empty after preprocessing)" if not tokens else ""))
print("Matches the stage-by-stage result:", final_tokens == stemmed)

## Part B · Train the bigram model

In [ ]:
training_sentences, stats = build_training_sentences(train_records)
model = BigramLanguageModel().fit(training_sentences)
print(f"Dataset                  : {info['parsed_rows']} parsed rows, {info['cleaning']['rows_out']} unique, "
      f"{info['train_messages']} training / {info['heldout_messages']} held-out (seed {info['seed']})")
print("Training sentences       :", model.number_of_training_sentences)
print("Vocabulary size V        :", model.vocabulary_size)
print("Distinct training words  :", len(model.word_counts))
print(f"C({START_TOKEN})                  :", model.context_counts[START_TOKEN])

## Part B · Step 1 · Chosen sentence and `<UNK>` replacement

In [ ]:
part_b_ready = False
if not sentences:
    print("No sentence to score: the message is empty.")
elif not 0 <= SENTENCE_INDEX < len(sentences):
    print(f"SENTENCE_INDEX={SENTENCE_INDEX} is out of range (valid: 0 to {len(sentences) - 1}).")
else:
    tokens = final_tokens[SENTENCE_INDEX]
    part_b_ready = True
    print(f"Scoring sentence [{SENTENCE_INDEX}] of {len(sentences)}: {sentences[SENTENCE_INDEX]!r}")
    print("Normalized tokens :", tokens)
    if not tokens:
        print("Note: empty after preprocessing; the model scores it as P(</s> | <s>).")
    mapped, replaced = model.replace_unseen_words(tokens)
    print("Unseen words -> <UNK>:", replaced if replaced else "none")
    print("Tokens after <UNK>   :", mapped)

## Part B · Step 2 · Padding with `<s>` and `</s>`

In [ ]:
if part_b_ready:
    calc = model.sentence_probability(tokens)
    padded = calc["padded_sentence"]
    print("Padded sentence:", padded)
else:
    print("Skipped: no valid sentence selected.")

## Part B · Step 3 · Bigrams

In [ ]:
if part_b_ready:
    pairs = list(nltk.bigrams(padded))
    print(f"{len(pairs)} bigrams:")
    for pair in pairs:
        print("  ", pair)
else:
    print("Skipped: no valid sentence selected.")

## Part B · Step 4 · Laplace-smoothed probability per bigram

In [ ]:
if part_b_ready:
    steps = [model.conditional_probability(previous, current) for previous, current in pairs]
    print(f"{'previous':<10}{'current':<10}{'C(prev,cur)':>12}{'C(prev)':>9}{'num':>6}{'den':>7}   "
          f"{'(a + 1) / (b + V) = x/y':<34}{'P':>10}")
    for s in steps:
        formula = (f"({s['bigram_count']} + 1) / ({s['previous_word_count']} + {s['vocabulary_size']}) = "
                   f"{s['numerator']}/{s['denominator']}")
        print(f"{s['previous_word']:<10}{s['current_word']:<10}{s['bigram_count']:>12}"
              f"{s['previous_word_count']:>9}{s['numerator']:>6}{s['denominator']:>7}   "
              f"{formula:<34}{s['probability']:>10.6f}")
else:
    print("Skipped: no valid sentence selected.")

## Part B · Step 5 · Running product

In [ ]:
if part_b_ready:
    running = 1.0
    for number, s in enumerate(steps, start=1):
        running *= s["probability"]
        print(f"{number:>2}. P({s['current_word']} | {s['previous_word']}) = {s['probability']:.6f}"
              f"   running product = {running:.4e}")
else:
    print("Skipped: no valid sentence selected.")

## Part B · Step 6 · Log probability (sum of logs)

In [ ]:
if part_b_ready:
    cumulative = 0.0
    for number, s in enumerate(steps, start=1):
        log_value = math.log(s["probability"])
        cumulative += log_value
        print(f"{number:>2}. ln P({s['current_word']} | {s['previous_word']}) = {log_value:.6f}"
              f"   cumulative sum = {cumulative:.6f}")
else:
    print("Skipped: no valid sentence selected.")

## Part B · Step 7 · Final result

In [ ]:
if part_b_ready:
    final = model.sentence_probability(tokens)
    print(f"P(sentence)          = {final['probability']:.10e}")
    print(f"ln P(sentence)       = {final['log_probability']:.6f}")
    print(f"exp(ln P(sentence))  = {math.exp(final['log_probability']):.10e}")
    print("Step-by-step product equals model result :", running == final["probability"])
    print("Step-by-step log sum equals model result :", math.isclose(cumulative, final["log_probability"]))
    print("exp(ln P) equals P                       :", math.isclose(
        math.exp(final["log_probability"]), final["probability"], rel_tol=1e-9))
else:
    print("Skipped: no valid sentence selected.")